In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully
Table VERSION_TRACKER created successfully
Table METRICS_TRACKER created successfully

Share anonymous install statistics? (opt-out instructions)

PixieDust will record metadata on its environment the next time the package is installed or updated. The data is anonymized and aggregated to help plan for future releases, and records only the following values:

{
   "data_sent": currentDate,
   "runtime": "python",
   "application_version": currentPixiedustVersion,
   "space_id": nonIdentifyingUniqueId,
   "config": {
       "repository_id": "https://github.com/ibm-watson-data-lab/pixiedust",
       "target_runtimes": ["Data Science Experience"],
       "event_id": "web",
       "event_organizer": "dev-journeys"
   }
}
You can opt out by calling pixiedust.optOut() in a new cell.


Pixiedust runtime updated. Please restart kernel
Table SPARK_PACKAGES created successfully
Table USER_PREFERENCES created successfully
Table service_connections created successfully
Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
     |████████████████████████████████| 20.2 MB 7.1 MB/s eta 0:00:01
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


Exception in thread Thread-6:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.7/threading.py", line 926, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.7/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 47, in startSparkJobProgressMonitor
    progressMonitor = SparkJobProgressMonitor()
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 174, in __init__
    self.addSparkListener()
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 203, in addSparkListener
    _env.getTemplate("sparkJobProgressMonitor/addSparkListener.scala").render()
  File "/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py", line 2352, in run_cell_magic
    result = fn(*args, **kwargs)
  File "</opt/conda/lib/python3.7/site-packages/decorator.py:d

In [8]:
scaled_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Smallset_scaleddf")

In [9]:
import numpy as np
# setting random seed for notebook reproducability
rnd_seed=23
np.random.seed=rnd_seed
np.random.set_state=rnd_seed

In [10]:
train_data, valid_data, test_data = scaled_df.randomSplit([.7,.2,.1], seed=rnd_seed)

In [11]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [12]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

Number of 0's in the label column: 70421
Number of 1's in the label column: 10662


In [13]:
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
count_zeros = 70421
count_ones = 10662

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced = upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame:  10590
Number of 1's in the balanced DataFrame:  70385


In [12]:
# from pyspark.ml.classification import GBTClassifier
# gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")

In [14]:
# from pyspark.ml.classification import GBTClassifier
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
# from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
# from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# # Set up the parameter grid for hyperparameter tuning
# paramGrid = {
#     "maxDepth": [5, 10],
#     "maxIter": [10, 20]
# }

# # Use BinaryClassificationEvaluator for model evaluation with AUC metric
# evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# # Store the best model and the best AUC score
# best_model = None
# best_auc = 0.0
# best_params = {}

# # Loop through all combinations of hyperparameters
# for maxDepth in paramGrid["maxDepth"]:
#     for maxIter in paramGrid["maxIter"]:
#         # Set hyperparameters
#         gbt.setMaxDepth(maxDepth)
#         gbt.setMaxIter(maxIter)

#         # Fit the model on the training data
#         model = gbt.fit(train_data_balanced)

#         # Evaluate the model on the validation data
#         valid_predictions = model.transform(valid_data)
#         validation_auc = evaluator.evaluate(valid_predictions)

#         # Print the current parameters and the validation AUC
#         print(f"MaxDepth: {maxDepth}, MaxIter: {maxIter}, Validation AUC: {validation_auc}")

#         # Check if this is the best model
#         if validation_auc > best_auc:
#             best_auc = validation_auc
#             best_model = model
#             best_params = {"maxDepth": maxDepth, "maxIter": maxIter}

# # Print the best parameters and validation AUC
# print(f"Best Parameters: {best_params}")
# print(f"Best Validation AUC: {best_auc}")

# # # Cast the label column in the test data to double (if necessary)
# # test_data = test_data.withColumn('label', test_data.label.cast('double'))

# # Make predictions on the test data using the best model found
# test_predictions = best_model.transform(test_data)

# # Prepare for evaluation by selecting label and prediction columns
# predictionAndTarget = test_predictions.select("label", "prediction")
# predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# # Convert the DataFrame to an RDD and compute the metrics
# predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# # Binary and Multiclass metrics
# metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
# metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# # Example: Print AUC and Accuracy
# print(f"Test AUC: {metrics_binary.areaUnderROC}")
# print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10, Validation AUC: 0.8510890525577335
MaxDepth: 5, MaxIter: 20, Validation AUC: 0.8649670179797417
MaxDepth: 10, MaxIter: 10, Validation AUC: 0.8575765337867054
MaxDepth: 10, MaxIter: 20, Validation AUC: 0.8673463986794264
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.8673463986794264
Test AUC: 0.5959782393648004
Test Accuracy: 0.5069120515647528


In [14]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
from pyspark.sql import DataFrame

# Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Function to perform hyperparameter tuning and model selection
def tune_gbt_model(train_data: DataFrame, valid_data: DataFrame, param_grid: dict, label_col: str):
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    
    best_model = None
    best_auc = 0.0
    best_params = {}

    # Loop through all combinations of hyperparameters
    for max_depth in param_grid["maxDepth"]:
        for max_iter in param_grid["maxIter"]:
            # Instantiate a new GBTClassifier for each hyperparameter combination
            gbt = GBTClassifier(labelCol=label_col, featuresCol="features_scaled", maxDepth=max_depth, maxIter=max_iter)

            # Fit the model on the training data
            model = gbt.fit(train_data_balanced)

            # Evaluate the model on the validation data
            valid_predictions = model.transform(valid_data)
            validation_auc = evaluator.evaluate(valid_predictions)

            # Print current parameters and validation AUC
            print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}, Validation AUC: {validation_auc}")

            # Check if this is the best model
            if validation_auc > best_auc:
                best_auc = validation_auc
                best_model = model
                best_params = {"maxDepth": max_depth, "maxIter": max_iter}

    return best_model, best_params, best_auc

# Call the function to tune the GBT model
best_model, best_params, best_auc = tune_gbt_model(train_data_balanced, valid_data, param_grid, label_col="label")

# Print the best parameters and validation AUC
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10, Validation AUC: 0.8481406275265163
MaxDepth: 5, MaxIter: 20, Validation AUC: 0.8635089915050647
MaxDepth: 10, MaxIter: 10, Validation AUC: 0.8569563902628453
MaxDepth: 10, MaxIter: 20, Validation AUC: 0.8674906676974757
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.8674906676974757
Test AUC: 0.5942612067158021
Test Accuracy: 0.5137816979051819


In [15]:
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 4469.0         FP: 73.0
Actual: 1     FN: 5741.0         TP: 1508.0
[[4469.   73.]
 [5741. 1508.]]

Metrics:
f1_1:  0.3415628539071348
f1_0:  0.6058839479392624
precision_1:  0.9538266919671095
precision_0:  0.4377081292850147
recall_1:  0.20802869361291212
recall_0:  0.9839277851166887
auc:  0.5959782393648004
accuracy:  0.5069120515647528
sensitivity:  0.20802869361291212
specificity:  0.9839277851166887


In [15]:
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 4571.0         FP: 94.0
Actual: 1     FN: 5639.0         TP: 1487.0
[[4571.   94.]
 [5639. 1487.]]

Metrics:
f1_1:  0.34156425864247153
f1_0:  0.6145882352941175
precision_1:  0.9405439595192916
precision_0:  0.44769833496571987
recall_1:  0.20867246702217232
recall_0:  0.9798499464094319
auc:  0.5942612067158021
accuracy:  0.5137816979051819
sensitivity:  0.20867246702217232
specificity:  0.9798499464094319


In [16]:
#############################################Balanced Df - with Cross Validation using GridSearchCV for hyperparam #############
# Set up the parameter grid for hyperparameter tuning
paramGrid = ParamGridBuilder() \
    .addGrid(gbt.maxDepth, [5, 10]) \
    .addGrid(gbt.maxIter, [10, 20]) \
    .build()

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Set up the CrossValidator
crossval = CrossValidator(estimator=gbt,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)  # 3-fold cross-validation

# Fit the model using cross-validation
cv_model = crossval.fit(train_data_balanced)

# Evaluate the best model on the validation data
valid_predictions = cv_model.transform(valid_data)
validation_auc = evaluator.evaluate(valid_predictions)
print(f"Best Validation AUC: {validation_auc}")

# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))

# Make predictions on the test data using the best model found by cross-validation
test_predictions = cv_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

Best Validation AUC: 0.8673463986794281
Test AUC: 0.5959782393648004
Test Accuracy: 0.5069120515647528


In [17]:
###################Using Balanced Samples after using GridSearchCV ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 4469.0         FP: 73.0
Actual: 1     FN: 5741.0         TP: 1508.0
[[4469.   73.]
 [5741. 1508.]]

Metrics:
f1_1:  0.3415628539071348
f1_0:  0.6058839479392624
precision_1:  0.9538266919671095
precision_0:  0.4377081292850147
recall_1:  0.20802869361291212
recall_0:  0.9839277851166887
auc:  0.5959782393648004
accuracy:  0.5069120515647528
sensitivity:  0.20802869361291212
specificity:  0.9839277851166887


# Testing with RandomSplit+ManualFunc####

In [1]:
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-smallset-finalF1_Numbered_NoNullStr.parquet")

In [2]:
balanced_df = Epilepsy_Combined.drop("personid")

In [3]:
import numpy as np
# setting random seed for notebook reproducability
rnd_seed=23
np.random.seed=rnd_seed
np.random.set_state=rnd_seed

In [4]:
train_data, valid_data, test_data = balanced_df.randomSplit([.7,.2,.1], seed=rnd_seed)

In [5]:
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [6]:
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

Getting VectorSizeHint for vector column: 'gender_onehot'
Column 'gender_onehot' vector size: 4
Getting VectorSizeHint for vector column: 'race_onehot'
Column 'race_onehot' vector size: 7
Final input columns for feature assembly: ['gender_onehot', 'race_onehot', 'age_of_TBI_diagnosis', 'MedicalHistory', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B

Pipeline fitting done.
Pipeline transformation done.


In [7]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [8]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

Number of 0's in the label column: 70414
Number of 1's in the label column: 10669


In [9]:
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
count_zeros = 70414
count_ones = 10669

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame:  10790
Number of 1's in the balanced DataFrame:  70468


In [11]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features_scaled", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_df)

            # Validate on validation set
            val_data_pred = model.transform(val_df)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = model  # Store the best model

    return results, best_model  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Extract positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation Accuracy={best_params[2]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation Accuracy=0.506, Validation AUC=0.8744
Train Accuracy: 0.9096
Train AUC: 0.9323
Validation Accuracy: 0.506
Validation AUC: 0.8744
Test Accuracy: 0.4989
Test AUC: 0.8735


In [ ]:
# # Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        print("Confusion Matrix Metrics:")
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

In [25]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features_scaled", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_data_balanced)

            # Validate on validation set
            val_data_pred = model.transform(valid_data)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = model  # Store the best model

    return results, best_model  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation Accuracy={best_params[2]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation Accuracy=0.4957, Validation AUC=0.8743
Train Accuracy: 0.9081
Train AUC: 0.9284
Validation Accuracy: 0.4957
Validation AUC: 0.8743
Test Accuracy: 0.4941
Test AUC: 0.8742


In [10]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features_scaled", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_data_balanced)

            # Validate on validation set
            val_data_pred = model.transform(valid_data)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation Accuracy={best_params[2]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation Accuracy=0.5102, Validation AUC=0.8815
Train Accuracy: 0.9118
Train AUC: 0.9344
Validation Accuracy: 0.5102
Validation AUC: 0.8815
Test Accuracy: 0.5075
Test AUC: 0.8742


In [11]:
# # Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        print("Confusion Matrix Metrics:")
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

Confusion Matrix Metrics:
True Positives: 1464
True Negatives: 4521
False Positives: 5741
False Negatives: 66
Accuracy: 0.5075
Precision 1: 0.2032
Recall 1: 0.9569
F1 Score 1: 0.3352
Precision 0: 0.9856
Recall 0: 0.4406
F1 Score 0: 0.6089
Specificity: 0.4406


# ManualSplit+Predef-Func############

In [12]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
import random
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, VectorSizeHint

# Step 0: Mimicking randomSplit with seed and randomness
def probabilistic_split(df: DataFrame, fractions: list, seed=None) -> list:
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)
    df_with_random = df.withColumn("random", random_col)
    splits = []
    prev_fraction = 0
    for fraction in cumulative_fractions:
        split_df = df_with_random.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
        splits.append(split_df.drop("random"))
        prev_fraction = fraction
    return splits

# Train, validation, and test sampling fractions
train_fraction = 0.7
valid_fraction = 0.2
test_fraction = 0.1
fractions = [train_fraction, valid_fraction, test_fraction]
seed_value = 23
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, seed=seed_value)

# Show class distribution in train, validation, and test datasets
train_data.groupBy('label').count().show()
valid_data.groupBy('label').count().show()
test_data.groupBy('label').count().show()

# Print counts
print("Sampled data count:", balanced_df.count())
print("Train data count:", train_data.count())
print("Validation data count:", valid_data.count())
print("Test data count:", test_data.count())

+-----+-----+
|label|count|
+-----+-----+
|  0.0|70345|
|  1.0|10738|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|20222|
|  1.0| 2975|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|10227|
|  1.0| 1565|
+-----+-----+

Sampled data count: 116072
Train data count: 81083
Validation data count: 23197
Test data count: 11792


In [13]:
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [14]:
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

Getting VectorSizeHint for vector column: 'gender_onehot'
Column 'gender_onehot' vector size: 4
Getting VectorSizeHint for vector column: 'race_onehot'
Column 'race_onehot' vector size: 7
Final input columns for feature assembly: ['gender_onehot', 'race_onehot', 'age_of_TBI_diagnosis', 'MedicalHistory', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B

Pipeline fitting done.
Pipeline transformation done.


In [15]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [16]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

Number of 0's in the label column: 70345
Number of 1's in the label column: 10738


In [17]:
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
count_zeros = 70345
count_ones = 10738

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame:  10552
Number of 1's in the balanced DataFrame:  70243


In [18]:
from pyspark.ml.classification import GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")

In [19]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for maxDepth in paramGrid["maxDepth"]:
    for maxIter in paramGrid["maxIter"]:
        # Set hyperparameters
        gbt.setMaxDepth(maxDepth)
        gbt.setMaxIter(maxIter)

        # Fit the model on the training data
        model = gbt.fit(train_data_balanced)

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)
        validation_auc = evaluator.evaluate(valid_predictions)

        # Print the current parameters and the validation AUC
        print(f"MaxDepth: {maxDepth}, MaxIter: {maxIter}, Validation AUC: {validation_auc}")

        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_model = model
            best_params = {"maxDepth": maxDepth, "maxIter": maxIter}

# Print the best parameters and validation AUC
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")

# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10, Validation AUC: 0.8498171722452219
MaxDepth: 5, MaxIter: 20, Validation AUC: 0.8647091569295117
MaxDepth: 10, MaxIter: 10, Validation AUC: 0.8587330214451518
MaxDepth: 10, MaxIter: 20, Validation AUC: 0.8725586826561291
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.8725586826561291
Test AUC: 0.5863947430608238
Test Accuracy: 0.4637890094979647


In [20]:
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 3983.0         FP: 79.0
Actual: 1     FN: 6244.0         TP: 1486.0
[[3983.   79.]
 [6244. 1486.]]

Metrics:
f1_1:  0.31974179666487357
f1_0:  0.557491776891315
precision_1:  0.949520766773163
precision_0:  0.3894592744695414
recall_1:  0.19223803363518757
recall_0:  0.9805514524864599
auc:  0.5863947430608238
accuracy:  0.4637890094979647
sensitivity:  0.19223803363518757
specificity:  0.9805514524864599
